# Feature Engineering Pipeline — From Raw Data

**Playground: seasonal_approach**

This notebook reproduces the full feature engineering pipeline from raw data only.
It consolidates the logic from `notebooks/01_dataset_exploration.ipynb` and
`notebooks/02_eda_preprocessing.ipynb`, with two upgrades:
- Uses the **new MTA subway file** (`dataset/subway nyc/`) which adds ADA, CBD, and route-count features
- Writes all outputs to `playground/seasonal_approach/data/` (nothing to `data/processed/`)

**Reads from (raw, read-only):**
- `data/raw/listings_2025-11-01.csv.gz` — November 2025 InsideAirbnb snapshot
- `data/raw/nyc_facilities.csv` — NYC POI/facilities
- `data/raw/nypd_complaint_data.csv` — NYPD arrests 2025
- `data/raw/lirr_stations.csv` — LIRR station locations
- `data/external/MTA_Subway_Stations_20260330.csv` — MTA subway stationsA_Subway_Stations_20260330.csv` — MTA stations (new, richer file)

**Writes to (playground only):**
- `playground/seasonal_approach/data/feature_matrix.csv`

---
## 0. Setup

In [1]:
import gzip
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Read-only source paths ---
REPO    = Path('../..')                       # team005-airbnb-nyc/
RAW     = REPO / 'data' / 'raw'
EXT     = Path('data/external')       # copied external datasets

# --- Playground output ---
OUT = Path('data')
OUT.mkdir(exist_ok=True)

# Verify key source files exist
sources = [
    RAW / 'listings_2025-11-01.csv.gz',
    RAW / 'nyc_facilities.csv',
    RAW / 'nypd_complaint_data.csv',
    RAW / 'lirr_stations.csv',
    EXT / 'MTA_Subway_Stations_20260330.csv',
]
for s in sources:
    status = '✓' if s.exists() else '✗ MISSING'
    print(f'{status}  {s.name}')

✓  listings_2025-11-01.csv.gz
✓  nyc_facilities.csv
✓  nypd_complaint_data.csv
✓  lirr_stations.csv
✓  MTA_Subway_Stations_20260330.csv


---
## 1. Spatial utility functions

Two functions used throughout:
- `nearest_distance` — for each listing, find distance in km to the nearest reference point
- `count_within_radius` — for each listing, count how many reference points fall within a radius

Both use chunked matrix operations (500 listings at a time) to avoid RAM overflow
when computing distances against tens of thousands of reference points.

In [2]:
def nearest_distance(listing_lats, listing_lons, ref_lats, ref_lons, chunk=500):
    """Distance in km from each listing to its nearest reference point."""
    R = 6371.0
    a1 = np.radians(listing_lats.values); o1 = np.radians(listing_lons.values)
    a2 = np.radians(ref_lats);            o2 = np.radians(ref_lons)
    min_d = np.full(len(a1), np.inf)
    for i in range(0, len(a1), chunk):
        la = a1[i:i+chunk, None]; lo = o1[i:i+chunk, None]
        dlat = a2[None,:] - la;   dlon = o2[None,:] - lo
        a = np.sin(dlat/2)**2 + np.cos(la)*np.cos(a2[None,:])*np.sin(dlon/2)**2
        min_d[i:i+chunk] = (R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))).min(axis=1)
    return min_d

def count_within_radius(listing_lats, listing_lons, ref_lats, ref_lons,
                        radius_km=0.5, chunk=500):
    """Count reference points within radius_km of each listing."""
    R = 6371.0
    a1 = np.radians(listing_lats.values); o1 = np.radians(listing_lons.values)
    a2 = np.radians(ref_lats);            o2 = np.radians(ref_lons)
    counts = np.zeros(len(a1), dtype=int)
    for i in range(0, len(a1), chunk):
        la = a1[i:i+chunk, None]; lo = o1[i:i+chunk, None]
        dlat = a2[None,:] - la;   dlon = o2[None,:] - lo
        a = np.sin(dlat/2)**2 + np.cos(la)*np.cos(a2[None,:])*np.sin(dlon/2)**2
        d = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        counts[i:i+chunk] = (d <= radius_km).sum(axis=1)
    return counts

def nearest_index(listing_lats, listing_lons, ref_lats, ref_lons, chunk=500):
    """Index of nearest reference point for each listing."""
    R = 6371.0
    a1 = np.radians(listing_lats.values); o1 = np.radians(listing_lons.values)
    a2 = np.radians(ref_lats);            o2 = np.radians(ref_lons)
    best_idx = np.zeros(len(a1), dtype=int)
    min_d = np.full(len(a1), np.inf)
    for i in range(0, len(a1), chunk):
        la = a1[i:i+chunk, None]; lo = o1[i:i+chunk, None]
        dlat = a2[None,:] - la;   dlon = o2[None,:] - lo
        a = np.sin(dlat/2)**2 + np.cos(la)*np.cos(a2[None,:])*np.sin(dlon/2)**2
        d = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        chunk_best = d.argmin(axis=1)
        chunk_min  = d.min(axis=1)
        # only update rows where this chunk beat the current best
        best_idx[i:i+chunk] = chunk_best
        min_d[i:i+chunk]    = chunk_min
    return best_idx

print('Spatial functions defined.')

Spatial functions defined.


---
## 2. Load and filter listings

Source: November 2025 InsideAirbnb snapshot.

**STR filter:** keep only listings where `minimum_nights < 28` and `price > 0`.
NYC Local Law 18 (effective Sept 2023) restricts short-term rentals to registered hosts
who are present during the stay — this is why only ~4k STR listings exist citywide.

In [3]:
with gzip.open(RAW / 'listings_2025-11-01.csv.gz', 'rt') as f:
    raw = pd.read_csv(f, low_memory=False)

print(f'Raw listings: {len(raw):,} rows, {raw.shape[1]} columns')

raw['price_numeric'] = (
    raw['price'].str.replace(r'[\$,]', '', regex=True)
    .pipe(pd.to_numeric, errors='coerce')
)
raw['min_nights_int'] = pd.to_numeric(raw['minimum_nights'], errors='coerce')

df = raw[
    (raw['min_nights_int'] < 28) &
    (raw['price_numeric'].notna()) &
    (raw['price_numeric'] > 0)
].copy()

print(f'After STR filter: {len(df):,} listings')
print(f'Price range: ${df["price_numeric"].min():.0f} – ${df["price_numeric"].quantile(0.99):.0f} (99th pct)')
print(f'Median price: ${df["price_numeric"].median():.0f}')
print()
print('Borough breakdown:')
print(df.groupby('neighbourhood_group_cleansed')['price_numeric'].agg(['count','median']).sort_values('median', ascending=False))

Raw listings: 36,353 rows, 79 columns
After STR filter: 4,073 listings
Price range: $23 – $50000 (99th pct)
Median price: $222

Borough breakdown:
                              count  median
neighbourhood_group_cleansed               
Manhattan                      1836   363.5
Brooklyn                       1364   176.0
Queens                          700   131.5
Bronx                           122   128.5
Staten Island                    51   109.0


---
## 3. MTA subway features

Using the new `MTA_Subway_Stations_20260330.csv` file (from `dataset/subway nyc/`),
which has richer columns than the old file in `data/raw/`.

**New features vs original pipeline:**
- `subway_lines_500m` — count of distinct subway route letters within 500m (transit connectivity)
- `nearest_subway_cbd` — is the nearest station flagged as Central Business District?
- `nearest_subway_ada` — is the nearest station ADA accessible? (0=no, 1=full, 2=partial)

The original features (`dist_subway_km`, `near_subway`) are retained.

In [4]:
subway = pd.read_csv(EXT / 'MTA_Subway_Stations_20260330.csv')
subway_clean = subway.dropna(subset=['GTFS Latitude', 'GTFS Longitude']).copy()
print(f'Subway stations: {len(subway_clean)} (of {len(subway)} total)')
print(f'Columns: {list(subway_clean.columns)}')

# --- Distance to nearest subway ---
print('\nComputing subway distances...')
df['dist_subway_km'] = nearest_distance(
    df['latitude'], df['longitude'],
    subway_clean['GTFS Latitude'].values,
    subway_clean['GTFS Longitude'].values
)
df['near_subway'] = (df['dist_subway_km'] <= 0.5).astype(int)

# --- Nearest station index (for CBD and ADA lookup) ---
nearest_idx = nearest_index(
    df['latitude'], df['longitude'],
    subway_clean['GTFS Latitude'].values,
    subway_clean['GTFS Longitude'].values
)
subway_clean = subway_clean.reset_index(drop=True)
df['nearest_subway_cbd'] = subway_clean.loc[nearest_idx, 'CBD'].values.astype(int)
df['nearest_subway_ada'] = subway_clean.loc[nearest_idx, 'ADA'].values

# --- Distinct subway routes within 500m ---
# Daytime Routes column is like 'N W' or 'A C E' — count distinct letters
print('Counting subway lines within 500m...')
RADIUS_500M = 0.5

R = 6371.0
a1 = np.radians(df['latitude'].values)
o1 = np.radians(df['longitude'].values)
a2 = np.radians(subway_clean['GTFS Latitude'].values)
o2 = np.radians(subway_clean['GTFS Longitude'].values)
routes_col = subway_clean['Daytime Routes'].fillna('').values

line_counts = np.zeros(len(df), dtype=int)
chunk = 200
for i in range(0, len(a1), chunk):
    la = a1[i:i+chunk, None]; lo = o1[i:i+chunk, None]
    dlat = a2[None,:] - la; dlon = o2[None,:] - lo
    a = np.sin(dlat/2)**2 + np.cos(la)*np.cos(a2[None,:])*np.sin(dlon/2)**2
    d = R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))  # (chunk, n_stations)
    within = d <= RADIUS_500M                           # bool mask
    for j, mask_row in enumerate(within):
        nearby_routes = ' '.join(routes_col[mask_row])
        line_counts[i+j] = len(set(nearby_routes.split()) - {''})

df['subway_lines_500m'] = line_counts

print(f'\ndist_subway_km: median={df["dist_subway_km"].median():.3f} km')
print(f'near_subway:    {df["near_subway"].mean()*100:.1f}% within 500m')
print(f'subway_lines_500m: median={df["subway_lines_500m"].median():.0f} lines')
print(f'nearest_subway_cbd: {df["nearest_subway_cbd"].mean()*100:.1f}% near CBD station')
print(f'nearest_subway_ada: value counts\n{df["nearest_subway_ada"].value_counts().sort_index()}')

Subway stations: 496 (of 496 total)
Columns: ['GTFS Stop ID', 'Station ID', 'Complex ID', 'Division', 'Line', 'Stop Name', 'Borough', 'CBD', 'Daytime Routes', 'Structure', 'GTFS Latitude', 'GTFS Longitude', 'North Direction Label', 'South Direction Label', 'ADA', 'ADA Northbound', 'ADA Southbound', 'ADA Notes', 'Georeference']

Computing subway distances...
Counting subway lines within 500m...

dist_subway_km: median=0.307 km
near_subway:    72.1% within 500m
subway_lines_500m: median=2 lines
nearest_subway_cbd: 35.2% near CBD station
nearest_subway_ada: value counts
nearest_subway_ada
0    2317
1    1624
2     132
Name: count, dtype: int64


---
## 4. NYC Facilities — POI density

In [5]:
fac = pd.read_csv(RAW / 'nyc_facilities.csv', low_memory=False)
fac_clean = fac.dropna(subset=['latitude', 'longitude']).copy()

tourist_domains = [
    'PARKS, GARDENS, AND HISTORICAL SITES',
    'LIBRARIES AND CULTURAL PROGRAMS',
    'HEALTH AND HUMAN SERVICES',
]
tourist_fac = fac_clean[fac_clean['facdomain'].isin(tourist_domains)]
print(f'All facilities with coords: {len(fac_clean):,}')
print(f'Tourist-relevant:           {len(tourist_fac):,}')

print('\nCounting POIs...')
df['poi_count_500m']  = count_within_radius(df['latitude'], df['longitude'],
                            fac_clean['latitude'].values, fac_clean['longitude'].values, 0.5)
df['poi_count_1km']   = count_within_radius(df['latitude'], df['longitude'],
                            fac_clean['latitude'].values, fac_clean['longitude'].values, 1.0)
df['tourist_poi_500m']= count_within_radius(df['latitude'], df['longitude'],
                            tourist_fac['latitude'].values, tourist_fac['longitude'].values, 0.5)

from scipy import stats
r, _ = stats.spearmanr(df['poi_count_500m'], df['price_numeric'])
print(f'poi_count_500m: median={df["poi_count_500m"].median():.0f},  Spearman r vs price={r:.3f}')

All facilities with coords: 34,708
Tourist-relevant:           12,167

Counting POIs...


poi_count_500m: median=91,  Spearman r vs price=0.468


---
## 5. NYPD crime density

In [6]:
crime = pd.read_csv(RAW / 'nypd_complaint_data.csv', low_memory=False)
crime_geo = crime.dropna(subset=['Latitude', 'Longitude']).copy()
print(f'NYPD arrests with coords: {len(crime_geo):,}')

df['crime_count_500m'] = count_within_radius(
    df['latitude'], df['longitude'],
    crime_geo['Latitude'].values, crime_geo['Longitude'].values, 0.5
)

r, _ = stats.spearmanr(df['crime_count_500m'], df['price_numeric'])
print(f'crime_count_500m: median={df["crime_count_500m"].median():.0f},  Spearman r vs price={r:.3f}')

NYPD arrests with coords: 278,953


crime_count_500m: median=562,  Spearman r vs price=0.288


---
## 6. LIRR proximity

In [7]:
lirr_raw = pd.read_csv(RAW / 'lirr_stations.csv')
lirr = lirr_raw[lirr_raw['Railroad'] == 'LIRR'].dropna(subset=['Latitude','Longitude'])
print(f'LIRR stations: {len(lirr)}')

df['dist_lirr_km']   = nearest_distance(df['latitude'], df['longitude'],
                           lirr['Latitude'].values, lirr['Longitude'].values)
df['lirr_count_1mi'] = count_within_radius(df['latitude'], df['longitude'],
                           lirr['Latitude'].values, lirr['Longitude'].values, 1.60934)

print(f'dist_lirr_km: median={df["dist_lirr_km"].median():.2f} km')

LIRR stations: 126
dist_lirr_km: median=1.96 km


---
## 7. Merge extra columns from raw listings

Re-open the compressed file to pull columns that weren't loaded initially
(amenities text blob, host stats, review scores, property type, occupancy estimates).
We only load what we need via `usecols` to keep memory manageable.

In [8]:
# All 79 columns are already in df from the initial full load above
# (amenities, property_type, bathrooms_text, host stats, review scores, etc.)
# The original notebooks needed a second merge because they loaded a subset first;
# here we load everything at once, so no merge is needed.

REQUIRED_COLS = [
    'property_type', 'bathrooms_text', 'amenities',
    'host_is_superhost', 'host_response_time', 'host_response_rate',
    'host_acceptance_rate', 'host_identity_verified',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    'instant_bookable', 'license',
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
]
missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    print(f'WARNING: missing columns: {missing_cols}')
else:
    print(f'All required columns present in df ({df.shape[1]} total columns) ✓')

All required columns present in df (92 total columns) ✓


---
## 8. Feature engineering

Convert raw columns into model-ready numeric features.

In [9]:
# --- Bathrooms ---
def parse_bathrooms(s):
    if pd.isna(s): return np.nan
    s = str(s).lower()
    if 'half' in s: return 0.5
    m = re.search(r'[\d\.]+', s)
    return float(m.group()) if m else np.nan

df['bathrooms'] = df['bathrooms_text'].apply(parse_bathrooms)
df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms'].median())
print('Bathrooms parsed. Median:', df['bathrooms'].median())

Bathrooms parsed. Median: 1.0


In [10]:
# --- Amenities ---
KEY_AMENITIES = {
    'has_wifi':         r'wifi|internet',
    'has_kitchen':      r'kitchen',
    'has_ac':           r'air conditioning',
    'has_gym':          r'gym|fitness',
    'has_parking':      r'parking',
    'has_pool':         r'pool',
    'has_washer':       r'washer',
    'has_dryer':        r'dryer',
    'has_elevator':     r'elevator',
    'has_doorman':      r'doorman',
    'has_pets_allowed': r'pets allowed',
}

def parse_amenities(s):
    if pd.isna(s): return 0, {k: 0 for k in KEY_AMENITIES}
    s_lower = s.lower()
    count = len(re.findall(r'"[^"]+"', s))
    flags = {k: int(bool(re.search(pat, s_lower))) for k, pat in KEY_AMENITIES.items()}
    return count, flags

parsed = df['amenities'].apply(parse_amenities)
df['amenity_count'] = parsed.apply(lambda x: x[0])
for key in KEY_AMENITIES:
    df[key] = parsed.apply(lambda x: x[1][key])

print('Amenity prevalence:')
print(df[list(KEY_AMENITIES)].mean().sort_values(ascending=False).round(3))

Amenity prevalence:


has_wifi            0.986
has_dryer           0.905
has_ac              0.757
has_kitchen         0.729
has_parking         0.683
has_washer          0.429
has_pets_allowed    0.266
has_elevator        0.247
has_gym             0.168
has_pool            0.033
has_doorman         0.000
dtype: float64


In [11]:
# --- Host features ---
def pct_to_float(s):
    if pd.isna(s): return np.nan
    return float(str(s).replace('%','').strip()) / 100.0

RESPONSE_TIME_MAP = {
    'within an hour': 1, 'within a few hours': 2,
    'within a day': 3,   'a few days or more': 4
}

df['host_response_rate_f']    = df['host_response_rate'].apply(pct_to_float)
df['host_acceptance_rate_f']  = df['host_acceptance_rate'].apply(pct_to_float)
df['host_response_time_ord']  = df['host_response_time'].str.lower().map(RESPONSE_TIME_MAP)
df['is_superhost']            = (df['host_is_superhost'] == 't').astype(int)
df['host_identity_verified_f']= (df['host_identity_verified'] == 't').astype(int)
df['is_instant_bookable']     = (df['instant_bookable'] == 't').astype(int)
df['has_license']             = df['license'].notna().astype(int)

for col in ['host_response_rate_f', 'host_acceptance_rate_f', 'host_response_time_ord']:
    df[col] = df[col].fillna(df[col].median())

print('Host features:')
print(df[['is_superhost','host_identity_verified_f','is_instant_bookable','has_license']].mean().round(3))

Host features:
is_superhost                0.505
host_identity_verified_f    0.905
is_instant_bookable         0.450
has_license                 0.991
dtype: float64


In [12]:
# --- Review scores ---
REVIEW_COLS = [
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value'
]
for col in REVIEW_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Numeric cols that need median fill
for col in ['bedrooms', 'beds', 'reviews_per_month',
            'estimated_occupancy_l365d', 'estimated_revenue_l365d']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].pipe(pd.to_numeric, errors='coerce').median())

print('Review scores median:', df[REVIEW_COLS].median().round(2).to_dict())

Review scores median: {'review_scores_rating': 4.83, 'review_scores_accuracy': 4.85, 'review_scores_cleanliness': 4.83, 'review_scores_checkin': 4.91, 'review_scores_communication': 4.91, 'review_scores_location': 4.8, 'review_scores_value': 4.73}


In [13]:
# --- Categorical encoding ---

# Room type → ordinal
ROOM_TYPE_MAP = {'Shared room': 0, 'Hotel room': 1, 'Private room': 2, 'Entire home/apt': 3}
df['room_type_ord'] = df['room_type'].map(ROOM_TYPE_MAP).fillna(1)

# Borough → one-hot
boro_dummies = pd.get_dummies(df['neighbourhood_group_cleansed'], prefix='boro')
df = pd.concat([df, boro_dummies], axis=1)

# Property type → simplified → one-hot
def simplify_property(s):
    if pd.isna(s): return 'Other'
    s = s.lower()
    if 'entire' in s: return 'Entire home'
    if 'private' in s: return 'Private room'
    if 'shared' in s: return 'Shared room'
    if 'hotel' in s: return 'Hotel room'
    return 'Other'

df['property_simplified'] = df['property_type'].apply(simplify_property)
prop_dummies = pd.get_dummies(df['property_simplified'], prefix='prop')
df = pd.concat([df, prop_dummies], axis=1)

print('Room type distribution:')
print(df['room_type'].value_counts())

Room type distribution:
room_type
Private room       2331
Entire home/apt    1535
Hotel room          165
Shared room          42
Name: count, dtype: int64


---
## 9. Assemble and save feature matrix

Select the final set of feature columns, add the log-price target, and save.
We keep `id`, `latitude`, `longitude`, `neighbourhood_cleansed`, and
`neighbourhood_group_cleansed` as metadata columns (not model inputs).

In [14]:
FEATURE_COLS = [
    # Property
    'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'room_type_ord', 'amenity_count', 'minimum_nights', 'availability_365',
    # Reviews & activity
    'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month',
    'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value',
    # Host
    'is_superhost', 'host_identity_verified_f', 'is_instant_bookable',
    'host_response_rate_f', 'host_acceptance_rate_f', 'host_response_time_ord',
    'has_license',
    # Amenity flags
    'has_wifi', 'has_kitchen', 'has_ac', 'has_gym', 'has_parking', 'has_pool',
    'has_washer', 'has_dryer', 'has_elevator', 'has_doorman', 'has_pets_allowed',
    # Spatial — subway (original + new)
    'dist_subway_km', 'near_subway', 'subway_lines_500m',
    'nearest_subway_cbd', 'nearest_subway_ada',
    # Spatial — LIRR
    'dist_lirr_km', 'lirr_count_1mi',
    # Spatial — POI and crime
    'poi_count_500m', 'poi_count_1km', 'tourist_poi_500m', 'crime_count_500m',
    # Occupancy estimates
    'estimated_occupancy_l365d', 'estimated_revenue_l365d',
]

# Add OHE columns dynamically
boro_cols = [c for c in df.columns if c.startswith('boro_')]
prop_cols = [c for c in df.columns if c.startswith('prop_')]
ALL_FEATURES = [c for c in FEATURE_COLS + boro_cols + prop_cols if c in df.columns]

# Final fill: any remaining NAs
for col in ALL_FEATURES:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Log-price target
df['log_price'] = np.log1p(df['price_numeric'])

# Assemble output
META = ['id', 'latitude', 'longitude', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed']
out_cols = META + ALL_FEATURES + ['price_numeric', 'log_price']
out_cols = [c for c in out_cols if c in df.columns]
feature_matrix = df[out_cols].copy()

missing = feature_matrix[ALL_FEATURES].isnull().sum().sum()
print(f'Feature matrix: {feature_matrix.shape}')
print(f'Feature columns: {len(ALL_FEATURES)}')
print(f'Missing values in features: {missing}')
print(f'\nNew subway features added vs original pipeline:')
print(f'  subway_lines_500m:    median={df["subway_lines_500m"].median():.0f}')
print(f'  nearest_subway_cbd:   {df["nearest_subway_cbd"].mean()*100:.1f}% near CBD')
print(f'  nearest_subway_ada:   {(df["nearest_subway_ada"]>0).mean()*100:.1f}% near accessible station')

Feature matrix: (4073, 66)
Feature columns: 59
Missing values in features: 0

New subway features added vs original pipeline:
  subway_lines_500m:    median=2
  nearest_subway_cbd:   35.2% near CBD
  nearest_subway_ada:   43.1% near accessible station


In [15]:
out_path = OUT / 'feature_matrix.csv'
feature_matrix.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {feature_matrix.shape}')
print(f'Size:  {out_path.stat().st_size / 1e6:.1f} MB')
print(f'\nFeature columns ({len(ALL_FEATURES)}):')
for i, c in enumerate(ALL_FEATURES, 1):
    print(f'  {i:2d}. {c}')

Saved: data/feature_matrix.csv
Shape: (4073, 66)
Size:  1.4 MB

Feature columns (59):
   1. accommodates
   2. bedrooms
   3. beds
   4. bathrooms
   5. room_type_ord
   6. amenity_count
   7. minimum_nights
   8. availability_365
   9. number_of_reviews
  10. number_of_reviews_ltm
  11. reviews_per_month
  12. review_scores_rating
  13. review_scores_accuracy
  14. review_scores_cleanliness
  15. review_scores_checkin
  16. review_scores_communication
  17. review_scores_location
  18. review_scores_value
  19. is_superhost
  20. host_identity_verified_f
  21. is_instant_bookable
  22. host_response_rate_f
  23. host_acceptance_rate_f
  24. host_response_time_ord
  25. has_license
  26. has_wifi
  27. has_kitchen
  28. has_ac
  29. has_gym
  30. has_parking
  31. has_pool
  32. has_washer
  33. has_dryer
  34. has_elevator
  35. has_doorman
  36. has_pets_allowed
  37. dist_subway_km
  38. near_subway
  39. subway_lines_500m
  40. nearest_subway_cbd
  41. nearest_subway_ada
  42. dist